# 🪴 Zen Focus: Gamificación para Deep Work

Bienvenido al tutorial interactivo de **Zen Focus**.

Cuando trabajamos en tareas de alta carga cognitiva, como calcular modelos de regresión lineal (OLS) o limpiar bases de datos complejas, los tiempos de ejecución pueden ser largos. Es en estas pausas donde solemos perder la concentración.

**Zen Focus** resuelve esto gamificando tu tiempo de espera con arte ASCII dinámico y bloqueando sitios web a nivel de sistema.

### Contenido de este notebook
| # | Sección | Feature |
|---|---------|----------|
| 1 | Instalación | — |
| 2 | Arquitectura | `TemaBase.info_clase()` |
| 3 | Explorar temas | `renderizar_ipython()` |
| 4 | Demo rápida | `SesionZen.demo()` |
| 5 | Decorador | `@con_progreso`  |
| 6 | Widget interactivo | `TemaWidget` |
| 7 | Sesión real en terminal | `SesionZen.iniciar()` |
| 8 | Escudo de red | `Escudo` (requiere sudo) |

# 1. Instalación de la librería


In [ ]:
!pip install --upgrade zen-focus rich ipywidgets --quiet

# Habilitar ipywidgets en Colab (necesario para TemaWidget)
try:
    from google.colab import output
    output.enable_custom_widget_manager()
except ImportError:
    pass  # No estamos en Colab, no es necesario


## 1. Arquitectura - TemaBase.info_clase()
Cualquier tema (`PlantaFlor`, `Cohete`, `Edificio`, `Bebida`) hereda de `TemaBase` y
debe implementar tres métodos abstractos: `evolucionar()`, `penalizar()` y `renderizar()`.

El método `info_clase()` muestra la jerarquía completa directamente en el notebook:

In [ ]:
from zen_focus.temas import TemaBase

# Ver la clase base abstracta
TemaBase.info_clase()

In [ ]:
from zen_focus.temas import PlantaFlor

# Ver una subclase concreta — nota cómo hereda de TemaBase
PlantaFlor.info_clase()

# 3. Explorar temas - renderizar_ipython()
El método original `renderizar()` devuelve un string con `\n` que Jupyter no interpreta
bien con `print()`. El nuevo método `renderizar_ipython()` muestra el arte como HTML
enriquecido sin bloquear el kernel ni necesitar `rich`.

In [ ]:
from zen_focus.temas import PlantaFlor, Cohete

mi_planta = PlantaFlor(nombre="Girasol Analítico")
mi_cohete = Cohete(mision="Apolo 11")

# Nivel 1 — estado inicial
mi_planta.renderizar_ipython()

In [ ]:
# Evolucionamos manualmente y vemos el cambio de estado
mi_planta.evolucionar()
mi_planta.evolucionar()
mi_planta.renderizar_ipython()  # nivel 3

In [ ]:
# El cohete también soporta el mismo método
mi_cohete.evolucionar()
mi_cohete.renderizar_ipython()

# 📖 Uso Básico
La librería puede usarse de dos maneras principales: integrada en tus scripts largos o como una sesión de enfoque dedicada.

### Demostración - SesionZen.demo(pasos)
`demo(pasos=N)` simula la sesión completa en N frames (default: 5),
sin esperar tiempo real. Ideal para pruebas antes de ser usado.

In [ ]:
from zen_focus.motor import SesionZen
from zen_focus.temas import Cohete

mi_cohete = Cohete(mision="Proyecto Final")
sesion = SesionZen(minutos=25, tema=mi_cohete)

# Muestra los 5 niveles de evolución en ~3 segundos
sesion.demo(pasos=5)

In [ ]:
# También funciona con PlantaFlor
from zen_focus.temas import PlantaFlor

sesion2 = SesionZen(minutos=50, tema=PlantaFlor("Bonsai de Datos"))
sesion2.demo(pasos=5)

## Decorador - @con_progreso
El decorador `@con_progreso` envuelve cualquier función de procesamiento de datos
y muestra el tema evolucionando mientras corre, **sin modificar el código interno**.

Soporta dos modos:
- **Automático**: el tema avanza en un hilo de fondo cada segundo.
- **Con pasos explícitos**: la función usa `yield` para señalar cuándo avanzar un nivel.

In [ ]:
import time
from zen_focus.decoradores import con_progreso
from zen_focus.temas import PlantaFlor

# ── Modo automático ───────────────────────────────────────────────────────
# El tema avanza solo mientras la función trabaja.

@con_progreso(tema=PlantaFlor("Limpieza de datos"), retardo=0.8)
def limpiar_dataset(n_filas: int):
    """Simula una limpieza de dataset costosa."""
    time.sleep(3)  # ← reemplaza con tu código real (pd.read_csv, dropna, etc.)
    return f"{n_filas} filas procesadas"

resultado = limpiar_dataset(100_000)
print(f"\nResultado: {resultado}")

In [ ]:
from zen_focus.temas import Cohete

# ── Modo con pasos explícitos (yield) ─────────────────────────────────────
# Cada yield avanza exactamente un nivel. Perfecto para pipelines por etapas.

@con_progreso(tema=Cohete("Pipeline ML"))
def pipeline_ml(X, y):
    """Pipeline de entrenamiento dividido en 4 etapas."""
    # Etapa 1: carga
    time.sleep(0.8)
    yield  # ← señal: etapa completada

    # Etapa 2: preprocesamiento
    time.sleep(0.8)
    yield

    # Etapa 3: entrenamiento
    time.sleep(0.8)
    yield

    # Etapa 4: evaluación
    time.sleep(0.8)
    yield

    return "accuracy: 0.94"  # valor de retorno normal

# Datos ficticios para la demo
X_demo = list(range(100))
y_demo = [x % 2 for x in X_demo]

resultado = pipeline_ml(X_demo, y_demo)
print(f"\nResultado del pipeline: {resultado}")

## Widget Interactivo - TemaWidget
`TemaWidget` convierte cualquier tema en un widget de `ipywidgets` con:
- **Slider** para explorar los niveles libremente
- **Botones** Evolucionar, Penalizar y Reset
- **Vista HTML** del arte ASCII actualizada en vivo

Útil para que los estudiantes interactúen con los objetos sin escribir código.

In [ ]:
from IPython.display import display
from zen_focus.widgets import TemaWidget
from zen_focus.temas import PlantaFlor

# Crear el widget y mostrarlo
w = TemaWidget(PlantaFlor("Demo Interactiva"))
display(w)


In [ ]:
from IPython.display import display
from zen_focus.temas import Cohete
from zen_focus.widgets import TemaWidget

w2 = TemaWidget(Cohete("Saturno V"))
display(w2)


# Sesión Real - SesionZen.iniciar() en la terminal
La sesión en tiempo real está diseñada para correrse **en terminal**, no en Jupyter
(bloquea el kernel durante la duración completa). Se muestra aquí solo como referencia.

In [ ]:
# Demo de 6 segundos (0.1 minutos) — solo para ver que funciona en notebook
from zen_focus.motor import SesionZen
from zen_focus.temas import PlantaFlor

sesion_corta = SesionZen(minutos=0.1, escudo=None, tema=PlantaFlor("Test"))

try:
    sesion_corta.iniciar()
except Exception as e:
    print(f"Nota: En Colab, rich.Live puede no renderizarse. Error: {e}")
    print("Usa sesion.demo() para notebooks — es el método recomendado.")

#
#
 
8
.
 
🛡
️
 
E
s
c
u
d
o
 
d
e
 
R
e
d


E
l
 
`
E
s
c
u
d
o
`
 
m
o
d
i
f
i
c
a
 
`
/
e
t
c
/
h
o
s
t
s
`
 
p
a
r
a
 
b
l
o
q
u
e
a
r
 
s
i
t
i
o
s
 
d
i
s
t
r
a
c
t
o
r
e
s
 
*
*
a
 
n
i
v
e
l
 
d
e
l
 
s
i
s
t
e
m
a
 
o
p
e
r
a
t
i
v
o
*
*
.


>
 
⚠
️
 
*
*
L
i
m
i
t
a
c
i
ó
n
 
e
n
 
C
o
l
a
b
 
/
 
J
u
p
y
t
e
r
H
u
b
:
*
*
 
e
l
 
e
s
c
u
d
o
 
m
o
d
i
f
i
c
a
 
e
l
 
a
r
c
h
i
v
o
 
h
o
s
t
s
 
d
e
l
 
*
c
o
n
t
e
n
e
d
o
r
 
d
e
l
 
s
e
r
v
i
d
o
r
*
,
 
n
o
 
d
e
l
 
n
a
v
e
g
a
d
o
r
 
d
e
l
 
e
s
t
u
d
i
a
n
t
e
.
 
P
a
r
a
 
b
l
o
q
u
e
o
 
r
e
a
l
 
d
e
 
d
i
s
t
r
a
c
c
i
o
n
e
s
,
 
e
j
e
c
u
t
a
 
d
e
s
d
e
 
t
u
 
t
e
r
m
i
n
a
l
 
l
o
c
a
l
 
c
o
n
 
`
s
u
d
o
`
.


#
#
#
 
C
ó
m
o
 
s
e
 
u
s
a
 
l
o
c
a
l
m
e
n
t
e
:


`
`
`
p
y
t
h
o
n

f
r
o
m
 
z
e
n
_
f
o
c
u
s
.
m
o
t
o
r
 
i
m
p
o
r
t
 
S
e
s
i
o
n
Z
e
n

f
r
o
m
 
z
e
n
_
f
o
c
u
s
.
e
s
c
u
d
o
 
i
m
p
o
r
t
 
E
s
c
u
d
o

f
r
o
m
 
z
e
n
_
f
o
c
u
s
.
t
e
m
a
s
 
i
m
p
o
r
t
 
C
o
h
e
t
e


d
i
s
t
r
a
c
c
i
o
n
e
s
 
=
 
[
"
t
w
i
t
t
e
r
.
c
o
m
"
,
 
"
i
n
s
t
a
g
r
a
m
.
c
o
m
"
,
 
"
y
o
u
t
u
b
e
.
c
o
m
"
]


#
 
E
l
 
c
o
n
t
e
x
t
 
m
a
n
a
g
e
r
 
g
a
r
a
n
t
i
z
a
 
r
e
s
t
a
u
r
a
r
 
e
l
 
i
n
t
e
r
n
e
t
 
a
u
n
q
u
e
 
h
a
y
a
 
e
r
r
o
r
e
s

w
i
t
h
 
E
s
c
u
d
o
(
b
l
o
q
u
e
a
r
=
d
i
s
t
r
a
c
c
i
o
n
e
s
)
 
a
s
 
m
i
_
e
s
c
u
d
o
:

 
 
 
 
m
i
s
i
o
n
 
=
 
C
o
h
e
t
e
(
m
i
s
i
o
n
=
"
A
p
o
l
o
 
1
1
"
)

 
 
 
 
s
e
s
i
o
n
 
=
 
S
e
s
i
o
n
Z
e
n
(
m
i
n
u
t
o
s
=
2
5
,
 
e
s
c
u
d
o
=
m
i
_
e
s
c
u
d
o
,
 
t
e
m
a
=
m
i
s
i
o
n
)

 
 
 
 
s
e
s
i
o
n
.
i
n
i
c
i
a
r
(
)

`
`
`


#
#
#
 
C
r
e
a
r
 
t
u
 
p
r
o
p
i
o
 
t
e
m
a
 
—
 
h
e
r
e
n
c
i
a
 
d
e
 
`
T
e
m
a
B
a
s
e
`
:


`
`
`
p
y
t
h
o
n

f
r
o
m
 
z
e
n
_
f
o
c
u
s
.
t
e
m
a
s
 
i
m
p
o
r
t
 
T
e
m
a
B
a
s
e


c
l
a
s
s
 
M
i
T
e
m
a
(
T
e
m
a
B
a
s
e
)
:

 
 
 
 
F
A
S
E
S
 
=
 
[
"
(
n
i
v
e
l
 
1
)
"
,
 
"
(
n
i
v
e
l
 
2
)
"
,
 
"
(
n
i
v
e
l
 
3
)
"
,
 
"
(
n
i
v
e
l
 
4
)
"
,
 
"
(
n
i
v
e
l
 
5
)
"
]


 
 
 
 
d
e
f
 
e
v
o
l
u
c
i
o
n
a
r
(
s
e
l
f
)
:

 
 
 
 
 
 
 
 
i
f
 
s
e
l
f
.
n
i
v
e
l
_
a
c
t
u
a
l
 
<
 
s
e
l
f
.
n
i
v
e
l
_
m
a
x
i
m
o
:

 
 
 
 
 
 
 
 
 
 
 
 
s
e
l
f
.
n
i
v
e
l
_
a
c
t
u
a
l
 
+
=
 
1


 
 
 
 
d
e
f
 
p
e
n
a
l
i
z
a
r
(
s
e
l
f
)
:

 
 
 
 
 
 
 
 
i
f
 
s
e
l
f
.
n
i
v
e
l
_
a
c
t
u
a
l
 
>
 
1
:

 
 
 
 
 
 
 
 
 
 
 
 
s
e
l
f
.
n
i
v
e
l
_
a
c
t
u
a
l
 
-
=
 
1


 
 
 
 
d
e
f
 
r
e
n
d
e
r
i
z
a
r
(
s
e
l
f
)
 
-
>
 
s
t
r
:

 
 
 
 
 
 
 
 
r
e
t
u
r
n
 
s
e
l
f
.
F
A
S
E
S
[
s
e
l
f
.
n
i
v
e
l
_
a
c
t
u
a
l
 
-
 
1
]


#
 
P
r
o
b
a
r
l
o
 
c
o
n
 
e
l
 
w
i
d
g
e
t

f
r
o
m
 
z
e
n
_
f
o
c
u
s
.
w
i
d
g
e
t
s
 
i
m
p
o
r
t
 
T
e
m
a
W
i
d
g
e
t

d
i
s
p
l
a
y
(
T
e
m
a
W
i
d
g
e
t
(
M
i
T
e
m
a
(
"
M
i
 
T
e
m
a
"
)
)
)